In [1]:
# 활성화함수: 모델이 비선형을 학습할 수 있도록 한다.
# - 다중 분류 모델 출력층: softmax(활성화함수)
# - 이진 분류 모델 출력층: sigmoid(활성화함수)
# - ReLU(활성화함수): 은닉층으로 입력이 0보다 크면 그대로 0보다 작으면 0을 출력
#  (기울기가 작아지지 않아 층을 깊게 쌓아도 학습이 가능함 → 깊은 층이 갖는 표현력을 실제로 활용할 수 있음)

# 딥러닝 모델 학습 과정
# 순전파: 입력으로부터 예측값을 계산
# 손실함수로 예측값과 실제값의 오차를 계산
# 역전파: 출력층에서 입력층으로 기울기 계산
# 계산된 기울기로 가중치 갱신(경사하강법=최적화)

Tensor는 PyTorch에서 데이터를 표현하는 기본 단위.
수학적으로는 다차원 배열(n차원 행렬)로 이해 가능.
- Vector = 1D Tensor
- Matrix = 2D Tensor
- nD Tensor = n차원

In [2]:
import torch
import numpy as np

In [3]:
arr = np.array([[1, 2, 3]])
t = torch.from_numpy(arr)

print("Tensor:", t)
print("Back to Numpy:", t.numpy())

# 주의: 메모리를 공유하므로 한 쪽 값을 바꾸면 다른 쪽도 바뀜

Tensor: tensor([[1, 2, 3]])
Back to Numpy: [[1 2 3]]


In [4]:
# 리스트로부터 생성
t1 = torch.tensor([1.0, 2.0, 3.0])
print("t1:", t1)

# 0으로 채워진 텐서
t2 = torch.zeros((2, 3))
print("t2:", t2)

# 1로 채워진 텐서
# t2b = torch.ones((2, 3))

# 평균 0, 표준편차 1의 정규분포 텐서
t3 = torch.randn((2, 2))
print("t3:", t3)

t1: tensor([1., 2., 3.])
t2: tensor([[0., 0., 0.],
        [0., 0., 0.]])
t3: tensor([[-0.1769, -0.0569],
        [ 0.7090,  1.9624]])


In [5]:
x = torch.rand((3, 4), dtype=torch.float32, device='cpu')

print("Shape:", x.shape)      # 크기와 차원
print("Data type:", x.dtype)  # 저장되는 데이터 타입
print("Device:", x.device)    # cpu인지 gpu인지

# x.to("cuda:0")  # GPU로 이동 (GPU 사용 가능할 때)

Shape: torch.Size([3, 4])
Data type: torch.float32
Device: cpu


In [6]:
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[10, 20], [30, 40]])

print("Add:\n", a + b)
print("Element-wise Multiply:\n", a * b)
print("Matrix Multiply:\n", a @ b)

# 함수 형태로도 가능
print("Add(func):\n", torch.add(a, b))
print("Matmul(func):\n", torch.matmul(a, b))

Add:
 tensor([[11, 22],
        [33, 44]])
Element-wise Multiply:
 tensor([[ 10,  40],
        [ 90, 160]])
Matrix Multiply:
 tensor([[ 70, 100],
        [150, 220]])
Add(func):
 tensor([[11, 22],
        [33, 44]])
Matmul(func):
 tensor([[ 70, 100],
        [150, 220]])


In [7]:
x = torch.randn(1, 3, 1, 5)
print("Original shape:", x.shape)

# 차원이 1인 부분 제거
x_squeezed = x.squeeze()
print("After squeeze:", x_squeezed.shape)

# 특정 위치에 차원 추가
x_unsqueezed = x_squeezed.unsqueeze(0)
print("After unsqueeze:", x_unsqueezed.shape)

# 모양 변경 (reshape), -1은 크기 자동 계산
x_viewed = x_squeezed.view(-1, 5)
print("After view:", x_viewed.shape)

Original shape: torch.Size([1, 3, 1, 5])
After squeeze: torch.Size([3, 5])
After unsqueeze: torch.Size([1, 3, 5])
After view: torch.Size([3, 5])


In [8]:
a = torch.tensor([[1], [2], [3]])   # 3x1
b = torch.tensor([40, 50, 60])      # 1x3

result = a + b  # 자동으로 3x3으로 확장되어 연산
print("Broadcasted result:\n", result)

Broadcasted result:
 tensor([[41, 51, 61],
        [42, 52, 62],
        [43, 53, 63]])


PyTorch Autograd: 연산 그래프를 활용해 자동으로 역전파(기울기 계산)를 수행하는 기능
- requires_grad=True인 Tensor는 연산 기록을 저장
- .backward() 호출 시 gradient가 자동 계산됨

In [ ]:
a = torch.tensor([3.0], requires_grad=True)
# a, b로 하는 계산 과정을 메모해둬 (나중에 "얼마나 바꿔야 할지" 계산하려고)
b = torch.tensor([2.0], requires_grad=True)

z = a * b + b ** 2  # z = ab + b^2
z.backward()

print("dz/da:", a.grad)  # = b = 2
print("dz/db:", b.grad)  # = a + 2b = 3 + 4 = 7

dz/da: tensor([2.])
dz/db: tensor([7.])


In [ ]:
x = torch.tensor([5.0], requires_grad=True)

with torch.no_grad():
  # 이 블록 안 계산은 메모하지 마 (얼마나 바꿔야 할지 계산 안 할 거니까)
  y = x ** 2  

print("Requires grad?", y.requires_grad)  # False
# 검증(inference) 시 기울기 계산 안 할 때 사용

Requires grad? False


In [ ]:
import torch.nn as nn

class MLP(nn.Module):
  def __init__(self):
    super(MLP, self).__init__()
    self.flatten = nn.Flatten()
    self.fc1 = nn.Linear(28*28, 128) # 선형 계산(가중치 내적과 bias 합하기)
    self.relu1 = nn.ReLU()
    self.fc2 = nn.Linear(128, 64)
    self.relu2 = nn.ReLU()
    self.fc3 = nn.Linear(64, 10)
    self.softmax = nn.Softmax(dim=1)

  def forward(self, x):
    x = self.flatten(x)
    x = self.relu1(self.fc1(x))
    x = self.relu2(self.fc2(x))
    x = self.softmax(self.fc3(x))
    return x

model = MLP()

In [ ]:
# Dropout = 학습 중 매 스텝마다 뉴런을 랜덤으로 일부 꺼버리는 기법
# 목적: 특정 뉴런에 과하게 의존하는 것(=과적합)을 막고 일반화 성능을 높임
# train() 때는 켜짐, eval() 때는 꺼짐 (실전에선 모든 뉴런을 다 써야 하니까)

model.train()
# "공부 모드" ON
# - Dropout 등 학습용 기능 다시 켜짐
# - 가중치들이 requires_grad=True로 설정됨 → 기울기 계산 + 갱신 가능
# 학습 루프(for epoch...) 안에서, 학습 데이터를 돌 때 호출

model.eval()
# "시험 모드" ON  
# - Dropout 등 학습용 기능 꺼짐 (모델이 흔들림 없이 일관되게 답함)
# - 가중치들이 requires_grad=False로 설정됨 → 기울기 계산 안 함
# 검증(validation) or 실전 예측(inference) 전에 반드시 호출

# 주의: eval()과 no_grad()는 역할이 다름
# - eval()      → Dropout 같은 "레이어 동작 방식"을 바꿈
# - no_grad()   → 기울기 "계산 자체"를 안 하게 메모리/속도 절약
# 그래서 검증할 땐 보통 이 둘을 같이 씀:
#   model.eval()
#   with torch.no_grad():
#     ...검증 코드...

In [ ]:
# 학습 Loop 순서:
# ① DataLoader에서 데이터와 라벨을 받아오고
# ② 최적화 함수를 초기화한 뒤 모델에 데이터를 입력하고
# ③ 예측값과 라벨값으로 손실을 계산해 역전파를 수행한 다음
# ④ 최적화 함수로 파라미터를 업데이트하는 순서로 진행되며
# 이를 모든 mini-batch에 대해 반복한다.

# epochs = 5

# for epoch in range(epochs):
#   model.train()
#   for images, labels in train_loader:
#     # ① DataLoader에서 데이터와 라벨을 받아옴 (images, labels)
#     optimizer.zero_grad()
#     # ② 최적화 함수 초기화
#     outputs = model(images)
#     # ② 모델에 데이터를 입력 (순전파)
#     loss = criterion(outputs, labels)
#     # ③ 예측값(outputs)과 라벨값(labels)으로 손실 계산
#     loss.backward()
#     # ③ 역전파 수행 (기울기 계산)
#     optimizer.step()
#     # ④ 최적화 함수로 파라미터(가중치) 업데이트
#     # → 이 for문이 모든 mini-batch에 대해 반복됨